In [18]:
# ================================================================
# SPENDWISE - EXPENSE TRACKER WITH LIVE USABILITY TRACKING
# Java backend + web app. Run this single cell in Google Colab.
# ================================================================

# >>> APNA NAAM YAHAN LIKHEIN <<<
USER_NAME = "AH"

import socket
import subprocess
import time
import urllib.request
from pathlib import Path

from IPython.display import display, HTML
from google.colab import output

PROJECT = Path("/content/SpendWiseApp")
PROJECT.mkdir(parents=True, exist_ok=True)
JAVA_FILE = PROJECT / "SpendWiseServer.java"
HTML_FILE = PROJECT / "index.html"
LOG_FILE = PROJECT / "server.log"

print("=" * 60)
print("SPENDWISE - EXPENSE TRACKER")
print("=" * 60)


# ---------------- Java check / install ----------------
def which(cmd):
    return subprocess.run(["bash", "-lc", "which " + cmd],
                          capture_output=True, text=True).stdout.strip()

java, javac = which("java"), which("javac")
if not java or not javac:
    print("Java compiler not found. Installing JDK (about 1 minute)...")
    subprocess.run(["bash", "-lc",
                    "apt-get update -qq && apt-get install -y -qq default-jdk"], check=True)
    java, javac = which("java"), which("javac")
print("Java :", java)
print("Javac:", javac)


# ================================================================
# JAVA BACKEND  (records every click with the user id)
# ================================================================
java_code = r'''
import com.sun.net.httpserver.HttpExchange;
import com.sun.net.httpserver.HttpHandler;
import com.sun.net.httpserver.HttpServer;

import java.io.ByteArrayOutputStream;
import java.io.File;
import java.io.IOException;
import java.io.InputStream;
import java.io.OutputStream;
import java.net.InetSocketAddress;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.text.SimpleDateFormat;
import java.util.Date;
import java.util.HashSet;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Map;
import java.util.Set;
import java.util.concurrent.CopyOnWriteArrayList;
import java.util.concurrent.Executors;

public class SpendWiseServer {

    private static final List<Event> events = new CopyOnWriteArrayList<Event>();
    private static volatile long sessionStart = System.currentTimeMillis();

    static class Event {
        String user, action, screen, element, type, time, x, y;
    }

    public static void main(String[] args) throws Exception {
        int port = Integer.parseInt(System.getProperty("tracker.port", "8080"));
        final String root = System.getProperty("tracker.root", ".");
        HttpServer server = HttpServer.create(new InetSocketAddress("127.0.0.1", port), 0);

        server.createContext("/", new HttpHandler() {
            public void handle(HttpExchange ex) throws IOException {
                File f = new File(root, "index.html");
                if (!f.exists()) { send(ex, "index.html not found", "text/plain", 404); return; }
                send(ex, new String(Files.readAllBytes(f.toPath()), StandardCharsets.UTF_8),
                     "text/html; charset=UTF-8", 200);
            }
        });

        server.createContext("/api/track", new HttpHandler() {
            public void handle(HttpExchange ex) throws IOException {
                String b = readBody(ex.getRequestBody());
                Event e = new Event();
                e.user = getValue(b, "user");
                e.action = getValue(b, "action");
                e.screen = getValue(b, "screen");
                e.element = getValue(b, "element");
                e.type = getValue(b, "type");
                e.x = getValue(b, "x");
                e.y = getValue(b, "y");
                e.time = new SimpleDateFormat("HH:mm:ss").format(new Date());
                if (e.action.length() == 0) e.action = "Unknown";
                if (e.type.length() == 0) e.type = "click";
                events.add(e);
                send(ex, "{\"success\":true}", "application/json", 200);
            }
        });

        server.createContext("/api/data", new HttpHandler() {
            public void handle(HttpExchange ex) throws IOException {
                Map<String, Integer> counts = new LinkedHashMap<String, Integer>();
                Set<String> users = new HashSet<String>();
                for (Event e : events) {
                    Integer old = counts.get(e.action);
                    counts.put(e.action, old == null ? 1 : old + 1);
                    if (e.user.length() > 0 && !e.user.equals("-")) users.add(e.user);
                }
                StringBuilder j = new StringBuilder("{");
                j.append("\"total\":").append(events.size()).append(",");
                j.append("\"unique\":").append(counts.size()).append(",");
                j.append("\"users\":").append(users.size()).append(",");
                j.append("\"seconds\":").append((System.currentTimeMillis() - sessionStart) / 1000).append(",");
                j.append("\"counts\":{");
                int i = 0;
                for (Map.Entry<String, Integer> en : counts.entrySet()) {
                    if (i++ > 0) j.append(",");
                    j.append("\"").append(escape(en.getKey())).append("\":").append(en.getValue());
                }
                j.append("},\"events\":").append(eventsToJson()).append("}");
                send(ex, j.toString(), "application/json", 200);
            }
        });

        server.createContext("/api/reset", new HttpHandler() {
            public void handle(HttpExchange ex) throws IOException {
                events.clear();
                sessionStart = System.currentTimeMillis();
                send(ex, "{\"success\":true}", "application/json", 200);
            }
        });

        server.createContext("/api/export", new HttpHandler() {
            public void handle(HttpExchange ex) throws IOException {
                send(ex, eventsToJson(), "application/json", 200);
            }
        });

        server.setExecutor(Executors.newFixedThreadPool(4));
        server.start();
        System.out.println("SERVER_STARTED PORT=" + port);
    }

    private static String eventsToJson() {
        StringBuilder j = new StringBuilder("[");
        int i = 0;
        for (Event e : events) {
            if (i++ > 0) j.append(",");
            j.append("{\"user\":\"").append(escape(e.user)).append("\",")
             .append("\"action\":\"").append(escape(e.action)).append("\",")
             .append("\"screen\":\"").append(escape(e.screen)).append("\",")
             .append("\"element\":\"").append(escape(e.element)).append("\",")
             .append("\"type\":\"").append(escape(e.type)).append("\",")
             .append("\"time\":\"").append(escape(e.time)).append("\",")
             .append("\"x\":\"").append(escape(e.x)).append("\",")
             .append("\"y\":\"").append(escape(e.y)).append("\"}");
        }
        return j.append("]").toString();
    }

    private static String readBody(InputStream in) throws IOException {
        ByteArrayOutputStream out = new ByteArrayOutputStream();
        byte[] buf = new byte[2048];
        int n;
        while ((n = in.read(buf)) != -1) out.write(buf, 0, n);
        return new String(out.toByteArray(), StandardCharsets.UTF_8);
    }

    private static String getValue(String json, String key) {
        if (json == null) return "";
        String search = "\"" + key + "\"";
        int k = json.indexOf(search);
        if (k < 0) return "";
        int colon = json.indexOf(":", k + search.length());
        if (colon < 0) return "";
        int s = colon + 1;
        while (s < json.length() && Character.isWhitespace(json.charAt(s))) s++;
        if (s >= json.length() || json.charAt(s) != '"') return "";
        s++;
        StringBuilder v = new StringBuilder();
        boolean esc = false;
        for (int i = s; i < json.length(); i++) {
            char c = json.charAt(i);
            if (esc) { v.append(c); esc = false; continue; }
            if (c == '\\') { esc = true; continue; }
            if (c == '"') break;
            v.append(c);
        }
        return v.toString();
    }

    private static String escape(String s) {
        if (s == null) return "";
        return s.replace("\\", "\\\\").replace("\"", "\\\"")
                .replace("\n", "\\n").replace("\r", "\\r").replace("\t", "\\t");
    }

    private static void send(HttpExchange ex, String content, String type, int status) throws IOException {
        byte[] data = content.getBytes(StandardCharsets.UTF_8);
        ex.getResponseHeaders().set("Content-Type", type);
        ex.getResponseHeaders().set("Cache-Control", "no-store");
        ex.sendResponseHeaders(status, data.length);
        OutputStream out = ex.getResponseBody();
        out.write(data);
        out.close();
    }
}
'''
JAVA_FILE.write_text(java_code, encoding="utf-8")


# ================================================================
# FRONTEND
# ================================================================
html_code = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>SpendWise - Expense Tracker</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
:root{--bg:#f3f5f9;--surface:#fff;--ink:#14213d;--muted:#64708a;--line:#e3e8f0;--brand:#0f766e;
  --brand-soft:#e0f4f1;--nav:#14213d;--danger:#c2410c;--ok:#15803d}
*{box-sizing:border-box}
body{margin:0;font-family:'Plus Jakarta Sans',system-ui,-apple-system,'Segoe UI',Arial,sans-serif;background:linear-gradient(135deg,#ffd6e8 0%,#ffffff 50%,#c9e4ff 100%) fixed;min-height:100vh;color:var(--ink)}
button,input,select{font:inherit}
button:focus-visible,input:focus-visible,select:focus-visible{outline:2px solid var(--brand);outline-offset:2px}
.hidden{display:none!important}

/* login */
.login{min-height:100vh;display:grid;place-items:center;padding:20px;background:linear-gradient(135deg,#ffd6e8 0%,#ffffff 50%,#c9e4ff 100%)}
.login-card{background:var(--surface);border-radius:14px;padding:32px;width:100%;max-width:400px}
.login-card h1{margin:14px 0 4px;font-size:24px}
.login-card p{margin:0 0 20px;color:var(--muted);font-size:14px}
.err{color:var(--danger);font-size:13px;min-height:18px;margin:-4px 0 10px}
.login-by{text-align:center;margin-top:18px;font-size:13px;color:var(--muted)}

/* header */
.header{height:64px;background:var(--nav);color:#fff;display:flex;align-items:center;justify-content:space-between;
  padding:0 24px;position:sticky;top:0;z-index:20}
.logo{font-weight:700;font-size:20px;display:flex;align-items:center;gap:10px}
.logo-mark{width:30px;height:30px;border-radius:8px;background:#2dd4bf;color:#14213d;display:grid;place-items:center;font-weight:700}
.head-right{display:flex;align-items:center;gap:14px}
.by{font-size:13px;color:#b8c4dc}.by b{color:#fff;font-weight:600}
.chip{background:rgba(255,255,255,.1);padding:6px 12px;border-radius:999px;font-size:13px}
.live{display:flex;align-items:center;gap:8px;font-size:13px;background:rgba(45,212,191,.14);color:#5eead4;padding:6px 12px;border-radius:999px}
.live i{width:8px;height:8px;border-radius:50%;background:#2dd4bf;animation:pulse 1.8s infinite}
@keyframes pulse{50%{opacity:.35}}
@media (prefers-reduced-motion:reduce){.live i{animation:none}}

/* layout */
.layout{display:flex;min-height:calc(100vh - 64px)}
.sidebar{width:220px;background:var(--surface);border-right:1px solid var(--line);padding:20px 12px;display:flex;flex-direction:column}
.nav-btn{width:100%;border:0;background:transparent;color:var(--ink);padding:11px 14px;margin-bottom:4px;text-align:left;
  border-radius:8px;cursor:pointer;font-size:14px;font-weight:500;display:flex;gap:10px;align-items:center}
.nav-btn:hover{background:var(--bg)}
.nav-btn.active{background:var(--brand-soft);color:var(--brand);font-weight:600}
.side-foot{margin-top:auto;font-size:12px;color:var(--muted);padding:12px 14px;border-top:1px solid var(--line)}
.side-foot b{color:var(--ink)}
.content{flex:1;padding:28px 32px;max-width:1200px}
.screen{display:none}.screen.active{display:block}
h1{margin:0 0 4px;font-size:26px;letter-spacing:-.3px}
.subtitle{color:var(--muted);margin:0 0 22px}

.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-bottom:20px}
.card,.panel{background:var(--surface);border:1px solid var(--line);border-radius:10px;box-shadow:0 1px 2px rgba(20,33,61,.06)}
.card{padding:18px}
.card-label{color:var(--muted);font-size:13px}
.card-number{font-size:26px;font-weight:700;margin-top:6px}
.panel{padding:20px;margin-bottom:18px}
.panel h2{margin:0 0 14px;font-size:16px}
.row2{display:grid;grid-template-columns:1fr 1fr;gap:18px}
.form-grid{display:grid;grid-template-columns:2fr 1fr 1fr auto;gap:10px;align-items:end}
label{display:block;font-size:12px;color:var(--muted);margin-bottom:5px;font-weight:600}
.buttons{display:flex;flex-wrap:wrap;gap:10px;align-items:center}
.btn{border:1px solid transparent;padding:10px 16px;border-radius:8px;cursor:pointer;font-weight:600;font-size:14px;transition:transform .1s,filter .15s}
.btn:hover{filter:brightness(.94)}.btn:active{transform:scale(.97)}
.primary{background:var(--brand);color:#fff}
.secondary{background:var(--surface);color:var(--ink);border-color:var(--line)}
.danger{background:#fff1ec;color:var(--danger);border-color:#fbd5c5}
.small{padding:6px 11px;font-size:13px}
.input{width:100%;padding:11px 13px;border:1px solid var(--line);border-radius:8px;background:var(--bg);color:var(--ink);margin-bottom:12px}
.form-grid .input{margin:0}
.list-item{display:flex;justify-content:space-between;align-items:center;gap:12px;padding:12px 14px;border:1px solid var(--line);border-radius:8px;margin-bottom:8px}
.list-item small{color:var(--muted);display:block;margin-top:2px}
.amount{font-weight:700;white-space:nowrap}
.empty{color:var(--muted);font-size:14px;padding:6px 0}
.tag{background:var(--brand-soft);color:var(--brand);border-radius:999px;padding:3px 10px;font-size:12px;font-weight:600}
.activity{max-height:300px;overflow-y:auto}
.activity-item{display:flex;justify-content:space-between;gap:10px;padding:10px 0;border-bottom:1px solid var(--line);font-size:14px}
.activity-item small{color:var(--muted);white-space:nowrap}
.bar-row{margin-bottom:12px;font-size:14px}
.bar-top{display:flex;justify-content:space-between;margin-bottom:5px}
.bar{height:8px;border-radius:999px;background:var(--bg);overflow:hidden}
.bar i{display:block;height:100%;background:var(--brand);border-radius:999px;transition:width .3s}
.bar.over i{background:var(--danger)}
.heatmap{position:relative;height:300px;border-radius:8px;border:1px solid var(--line);overflow:hidden;background-color:var(--bg);
  background-image:linear-gradient(var(--line) 1px,transparent 1px),linear-gradient(90deg,var(--line) 1px,transparent 1px);background-size:30px 30px}
.dot{position:absolute;width:16px;height:16px;border-radius:50%;transform:translate(-50%,-50%);background:rgba(234,88,12,.55);box-shadow:0 0 0 6px rgba(234,88,12,.15)}
.toolbar{display:flex;justify-content:space-between;align-items:center;gap:12px;margin-bottom:14px;flex-wrap:wrap}
.toolbar select{width:auto;margin:0}
.toast{position:fixed;right:20px;bottom:20px;background:var(--nav);color:#fff;padding:12px 18px;border-radius:8px;font-size:14px;
  opacity:0;transform:translateY(10px);transition:all .25s;z-index:60;pointer-events:none}
.toast.show{opacity:1;transform:none}
.footer{color:var(--muted);font-size:13px;text-align:center;padding:10px 0 24px}

@media(max-width:850px){
  .sidebar{width:64px;padding:16px 8px}.nav-btn{justify-content:center;padding:12px 0}.nav-btn span,.side-foot{display:none}
  .cards{grid-template-columns:repeat(2,1fr)}.row2{grid-template-columns:1fr}.form-grid{grid-template-columns:1fr 1fr}.by{display:none}
}
@media(max-width:550px){.cards{grid-template-columns:1fr}.content{padding:18px}.form-grid{grid-template-columns:1fr}}
</style>
</head>
<body>

<!-- LOGIN -->
<div class="login" id="loginView">
  <div class="login-card">
    <div class="logo-mark" style="background:#0f766e;color:#fff;width:42px;height:42px;font-size:20px;border-radius:10px;display:grid;place-items:center;font-weight:700">S</div>
    <h1>Sign in to SpendWise</h1>
    <p>Track your daily expenses. Your data is saved under your User ID.</p>
    <label for="uid">User ID</label>
    <input id="uid" class="input" placeholder="e.g. ali_123" autocomplete="off">
    <label for="uname">Your name</label>
    <input id="uname" class="input" placeholder="e.g. Ali Khan" autocomplete="off">
    <div class="err" id="loginErr"></div>
    <button class="btn primary" id="loginBtn" style="width:100%">Sign in</button>
    <div class="login-by">Developed by <b class="dev">__USER_NAME__</b></div>
  </div>
</div>

<!-- APP -->
<div id="appView" class="hidden">
<header class="header">
  <div class="logo"><div class="logo-mark">S</div>SpendWise</div>
  <div class="head-right">
    <div class="by">Developed by <b class="dev">__USER_NAME__</b></div>
    <div class="chip" id="userChip"></div>
    <div class="live"><i></i>Tracking on</div>
    <button class="btn secondary small" id="logoutBtn">Sign out</button>
  </div>
</header>

<div class="layout">
<aside class="sidebar">
  <button class="nav-btn active" data-screen="dashboard">&#127968;<span>Dashboard</span></button>
  <button class="nav-btn" data-screen="expenses">&#128179;<span>Expenses</span></button>
  <button class="nav-btn" data-screen="budget">&#127919;<span>Budget</span></button>
  <button class="nav-btn" data-screen="analytics">&#128202;<span>Analytics</span></button>
  <div class="side-foot">Developed by<br><b class="dev">__USER_NAME__</b></div>
</aside>

<main class="content">

<section id="dashboard" class="screen active">
  <h1>Hello, <span id="helloName"></span></h1>
  <p class="subtitle">Add an expense below. Your selected budget period updates instantly, and expenses that exceed the limit are not added.</p>
  <div class="cards">
    <div class="card"><div class="card-label">Total spent</div><div class="card-number" id="sTotal">Rs 0</div></div>
    <div class="card"><div class="card-label">Budget left</div><div class="card-number" id="sLeft">-</div></div>
    <div class="card"><div class="card-label">Expenses</div><div class="card-number" id="sCount">0</div></div>
    <div class="card"><div class="card-label">Top category</div><div class="card-number" id="sTop">-</div></div>
  </div>
  <div class="panel">
    <h2>Add expense</h2>
    <div class="form-grid">
      <div><label for="eTitle">What did you spend on?</label><input id="eTitle" class="input" placeholder="Lunch, bus fare..."></div>
      <div><label for="eAmount">Amount (Rs)</label><input id="eAmount" class="input" type="number" min="1" placeholder="500"></div>
      <div><label for="eCat">Category</label><select id="eCat" class="input"></select></div>
      <button class="btn primary" id="addExpense">Add expense</button>
    </div>
  </div>
  <div class="row2">
    <div class="panel"><h2>Latest expenses</h2><div id="recentExp"></div></div>
    <div class="panel"><h2>Live activity</h2><div id="recentActivity" class="activity"><div class="empty">No activity yet.</div></div></div>
  </div>
</section>

<section id="expenses" class="screen">
  <h1>All expenses</h1>
  <p class="subtitle">Filter, delete or download your expense history.</p>
  <div class="panel">
    <div class="toolbar">
      <select id="filterCat" class="input"></select>
      <div class="buttons">
        <button class="btn secondary" id="csvBtn">Download CSV</button>
        <button class="btn danger" id="clearBtn">Clear all</button>
      </div>
    </div>
    <div id="expList"></div>
  </div>
</section>

<section id="budget" class="screen">
  <h1>Budget</h1>
  <p class="subtitle">Set your budgets all together below. Leave any box empty to skip that specific budget limit.</p>
  <div class="row2">
    <div class="panel">
      <h2>Set all budgets</h2>
      <div style="display:grid; grid-template-columns: 1fr 1fr; gap: 12px; align-items: center; margin-bottom: 12px;">
        <div>
          <label for="b_total" style="margin-bottom:2px;">Total budget</label>
          <span style="font-size:11px; color:var(--muted);">Overall limit</span>
        </div>
        <input id="b_total" class="input" type="number" min="0" placeholder="e.g. 50000" style="margin:0;">

        <div>
          <label for="b_monthly" style="margin-bottom:2px;">Monthly budget</label>
          <span style="font-size:11px; color:var(--muted);">This month limit</span>
        </div>
        <input id="b_monthly" class="input" type="number" min="0" placeholder="e.g. 20000" style="margin:0;">

        <div>
          <label for="b_weekly" style="margin-bottom:2px;">Weekly budget</label>
          <span style="font-size:11px; color:var(--muted);">This week limit</span>
        </div>
        <input id="b_weekly" class="input" type="number" min="0" placeholder="e.g. 5000" style="margin:0;">

        <div>
          <label for="b_daily" style="margin-bottom:2px;">Daily budget</label>
          <span style="font-size:11px; color:var(--muted);">Today limit</span>
        </div>
        <input id="b_daily" class="input" type="number" min="0" placeholder="e.g. 800" style="margin:0;">
      </div>
      <button class="btn primary" id="saveBudget" style="width:100%; margin-top:6px;">Save budgets</button>
    </div>
    <div class="panel">
      <h2>Budget used</h2>
      <div id="budgetInfo"><div class="empty">Set a budget to see progress.</div></div>
    </div>
  </div>
  <div class="panel"><h2>Spending by category</h2><div id="catBars"></div></div>
</section>

<section id="analytics" class="screen">
  <h1>Analytics</h1>
  <p class="subtitle">Every click and task is recorded live by the Java server.</p>
  <div class="cards">
    <div class="card"><div class="card-label">Total clicks</div><div class="card-number" id="totalClicks">0</div></div>
    <div class="card"><div class="card-label">Unique actions</div><div class="card-number" id="uniqueActions">0</div></div>
    <div class="card"><div class="card-label">Users</div><div class="card-number" id="userCount">0</div></div>
    <div class="card"><div class="card-label">Session time</div><div class="card-number" id="sessionTime">0s</div></div>
  </div>
  <div class="row2">
    <div class="panel"><h2>Tasks performed (clicks per action)</h2><div id="actionCounts"><div class="empty">No actions recorded yet.</div></div></div>
    <div class="panel"><h2>Click map</h2><div id="heatmap" class="heatmap"></div></div>
  </div>
  <div class="panel"><h2>Recent activity</h2><div id="activityFull" class="activity"><div class="empty">No activity yet.</div></div></div>
  <div class="panel">
    <h2>Data management</h2>
    <div class="buttons">
      <button id="exportBtn" class="btn primary">Export JSON</button>
      <button id="resetBtn" class="btn danger">Reset tracking</button>
    </div>
  </div>
</section>

<div class="footer">SpendWise &middot; Developed by <b class="dev">__USER_NAME__</b></div>
</main>
</div>
</div>

<div class="toast" id="toast"></div>

<script>
var $ = function(i){ return document.getElementById(i); };
var CATS = ["Food","Transport","Shopping","Bills","Health","Other"];
var BTYPES = ["total", "monthly", "weekly", "daily"];
var user = null, data = {expenses: [], budgets: {total: 0, monthly: 0, weekly: 0, daily: 0}}, lastEvt = null;
document.addEventListener("click", function(e){ lastEvt = e; }, true);

function lsGet(k, f){ try { var v = localStorage.getItem(k); return v ? JSON.parse(v) : f; } catch(e){ return f; } }
function lsSet(k, v){ try { localStorage.setItem(k, JSON.stringify(v)); } catch(e){} }
function money(n){ return "Rs " + Number(n).toLocaleString(); }
function toast(m){ var t = $("toast"); t.textContent = m; t.classList.add("show");
  clearTimeout(toast.h); toast.h = setTimeout(function(){ t.classList.remove("show"); }, 2200); }
function el(tag, cls, text){ var n = document.createElement(tag); if (cls) n.className = cls; if (text !== undefined) n.textContent = text; return n; }

/* ---------- tracking: every click goes to the Java server ---------- */
function track(action, type, element){
  var s = document.querySelector(".screen.active");
  var x = "50", y = "50";
  if (lastEvt && lastEvt.clientX !== undefined){
    x = String(Math.max(0, Math.min(100, Math.round(lastEvt.clientX / window.innerWidth * 100))));
    y = String(Math.max(0, Math.min(100, Math.round(lastEvt.clientY / window.innerHeight * 100))));
  }
  fetch("/api/track", {method: "POST", headers: {"Content-Type": "application/json"},
    body: JSON.stringify({user: user ? user.id : "-", action: action,
      screen: user ? (s ? s.id : "") : "login", element: element || "", type: type || "click", x: x, y: y})
  }).then(loadAnalytics).catch(function(){});
}

/* ---------- login ---------- */
function showApp(){
  $("loginView").classList.add("hidden"); $("appView").classList.remove("hidden");
  $("userChip").textContent = user.name + " (" + user.id + ")";
  $("helloName").textContent = user.name;
  data = lsGet("sw:data:" + user.id, {expenses: [], budgets: {total: 0, monthly: 0, weekly: 0, daily: 0}});
  normalizeData();
  renderAll(); loadAnalytics();
}
$("loginBtn").onclick = function(){
  var id = $("uid").value.trim().toLowerCase(), name = $("uname").value.trim();
  if (!/^[a-z0-9_.-]{3,20}$/.test(id)){
    $("loginErr").textContent = "User ID must be 3-20 letters, numbers, dot, dash or underscore."; return;
  }
  $("loginErr").textContent = "";
  user = {id: id, name: name || id};
  lsSet("sw:session", user);
  showApp(); track("Login", "auth", "Sign in");
};
$("uname").addEventListener("keydown", function(e){ if (e.key === "Enter") $("loginBtn").click(); });
$("uid").addEventListener("keydown", function(e){ if (e.key === "Enter") $("uname").focus(); });
$("logoutBtn").onclick = function(){
  track("Logout", "auth", "Sign out");
  lsSet("sw:session", null); user = null;
  $("appView").classList.add("hidden"); $("loginView").classList.remove("hidden");
  $("uid").value = ""; $("uname").value = "";
};

/* ---------- navigation ---------- */
function go(screen){
  document.querySelectorAll(".screen").forEach(function(s){ s.classList.remove("active"); });
  $(screen).classList.add("active");
  document.querySelectorAll(".nav-btn").forEach(function(n){ n.classList.toggle("active", n.getAttribute("data-screen") === screen); });
  track("Open " + screen, "navigation", "sidebar");
}
document.querySelectorAll(".nav-btn").forEach(function(b){ b.onclick = function(){ go(b.getAttribute("data-screen")); }; });

/* ---------- expenses (the real work of the app) ---------- */
function save(){ lsSet("sw:data:" + user.id, data); }

function normalizeData(){
  if (!Array.isArray(data.expenses)) data.expenses = [];
  if (!data.budgets || typeof data.budgets !== "object") data.budgets = {};
  if (typeof data.budget === "number") data.budget = {amount: data.budget, type: "monthly"};
  if (data.budget && typeof data.budget === "object" && data.budget.amount > 0 &&
      BTYPES.indexOf(data.budget.type) >= 0 && !(data.budgets[data.budget.type] > 0)){
    data.budgets[data.budget.type] = data.budget.amount;
  }
  delete data.budget;
  BTYPES.forEach(function(t){ if (!(data.budgets[t] >= 0)) data.budgets[t] = 0; });

  data.expenses.forEach(function(e){
    if (!e.ts){
      var parsed = Date.parse(e.d || "");
      e.ts = isNaN(parsed) ? Date.now() : parsed;
    }
  });
}

function budgetLabel(type){
  return type === "daily" ? "Daily budget" :
         type === "weekly" ? "Weekly budget" :
         type === "total" ? "Total budget" : "Monthly budget";
}

function inBudgetPeriod(ts, type){
  if (type === "total") return true;
  var d = new Date(ts), now = new Date();

  if (type === "daily"){
    return d.getFullYear() === now.getFullYear() &&
           d.getMonth() === now.getMonth() &&
           d.getDate() === now.getDate();
  }

  if (type === "monthly"){
    return d.getFullYear() === now.getFullYear() &&
           d.getMonth() === now.getMonth();
  }

  var start = new Date(now.getFullYear(), now.getMonth(), now.getDate());
  var day = start.getDay();
  var diff = day === 0 ? 6 : day - 1;
  start.setDate(start.getDate() - diff);
  start.setHours(0,0,0,0);

  var end = new Date(start);
  end.setDate(start.getDate() + 7);
  return d >= start && d < end;
}

function periodTotals(type){
  var t = 0, by = {};
  data.expenses.forEach(function(e){
    if (!inBudgetPeriod(e.ts || Date.now(), type)) return;
    t += e.a;
    by[e.c] = (by[e.c] || 0) + e.a;
  });
  return {total: t, by: by};
}

function expenseRow(e, withDelete){
  var row = el("div", "list-item");
  var left = el("div"); left.appendChild(el("b", "", e.t));
  left.appendChild(el("small", "", e.c + " · " + e.d));
  var right = el("div", "buttons"); right.appendChild(el("span", "amount", money(e.a)));
  if (withDelete){
    var d = el("button", "btn danger small", "Delete");
    d.onclick = function(){
      data.expenses = data.expenses.filter(function(x){ return x.id !== e.id; });
      save(); renderAll(); toast("Expense deleted"); track("Delete Expense", "button", e.c);
    };
    right.appendChild(d);
  }
  row.appendChild(left); row.appendChild(right); return row;
}

function renderAll(){
  var period = periodTotals("total");
  var keys = Object.keys(period.by);

  $("sTotal").textContent = money(period.total);
  $("sCount").textContent = data.expenses.length;
  var left = null;
  BTYPES.forEach(function(t){
    var lim = Number(data.budgets[t]) || 0;
    if (lim > 0){ var r = lim - periodTotals(t).total; if (left === null || r < left) left = r; }
  });
  $("sLeft").textContent = left === null ? "-" : money(left);
  $("sTop").textContent = keys.length ? keys.sort(function(a, b){ return period.by[b] - period.by[a]; })[0] : "-";

  var recent = $("recentExp"); recent.innerHTML = "";
  if (!data.expenses.length) recent.appendChild(el("div", "empty", "No expenses yet. Add your first one above."));
  data.expenses.slice(-5).reverse().forEach(function(e){ recent.appendChild(expenseRow(e, false)); });

  var f = $("filterCat").value, list = $("expList"); list.innerHTML = "";
  var shown = data.expenses.filter(function(e){ return !f || f === "All" || e.c === f; });
  if (!shown.length) list.appendChild(el("div", "empty", "No expenses to show."));
  shown.slice().reverse().forEach(function(e){ list.appendChild(expenseRow(e, true)); });

  BTYPES.forEach(function(t){
    var inp = $("b_" + t);
    if (inp && document.activeElement !== inp) inp.value = data.budgets[t] || "";
  });
  var info = $("budgetInfo"); info.innerHTML = "";
  var anySet = false;
  BTYPES.forEach(function(t){
    var lim = Number(data.budgets[t]) || 0;
    if (!(lim > 0)) return;
    anySet = true;
    var spent = periodTotals(t).total, raw = spent / lim * 100;
    var r = el("div", "bar-row"), top = el("div", "bar-top");
    top.appendChild(el("span", "", budgetLabel(t)));
    top.appendChild(el("span", "tag", money(spent) + " / " + money(lim) + " (" + Math.round(raw) + "%)"));
    var b = el("div", "bar"), i = el("i");
    i.style.width = Math.min(100, Math.round(raw)) + "%";
    if (spent > lim) b.classList.add("over");
    b.appendChild(i); r.appendChild(top); r.appendChild(b); info.appendChild(r);
  });
  if (!anySet) info.appendChild(el("div", "empty", "Set a budget to see progress."));

  var cb = $("catBars"); cb.innerHTML = "";
  if (!keys.length) cb.appendChild(el("div", "empty", "No spending recorded for this budget period."));
  keys.sort(function(a, b){ return period.by[b] - period.by[a]; }).forEach(function(k){
    var r = el("div", "bar-row"), top = el("div", "bar-top");
    top.appendChild(el("span", "", k)); top.appendChild(el("span", "tag", money(period.by[k])));
    var b = el("div", "bar"), i = el("i");
    i.style.width = period.total ? (period.by[k] / period.total * 100) + "%" : "0%";
    b.appendChild(i); r.appendChild(top); r.appendChild(b); cb.appendChild(r);
  });
}

$("addExpense").onclick = function(){
  var t = $("eTitle").value.trim(), a = parseFloat($("eAmount").value), c = $("eCat").value;
  if (!t){ toast("Enter what you spent on"); $("eTitle").focus(); return; }
  if (!(a > 0)){ toast("Enter an amount greater than 0"); $("eAmount").focus(); return; }

  var blocked = null;
  BTYPES.forEach(function(t){
    var lim = Number(data.budgets[t]) || 0;
    if (!(lim > 0)) return;
    var rem = Math.max(0, lim - periodTotals(t).total);
    if (a > rem && (!blocked || rem < blocked.rem)) blocked = {type: t, rem: rem};
  });
  if (blocked){
    toast("Not added: " + budgetLabel(blocked.type) + " exceeded. Only " + money(blocked.rem) + " is available.");
    track("Expense Blocked - Budget Exceeded", "validation", c);
    return;
  }

  var now = Date.now();
  data.expenses.push({
    id: now,
    t: t,
    a: a,
    c: c,
    d: new Date(now).toLocaleDateString(),
    ts: now
  });
  save(); renderAll(); $("eTitle").value = ""; $("eAmount").value = "";
  toast("Added " + money(a) + " to " + c); track("Add Expense", "form", c);
};

$("eAmount").addEventListener("keydown", function(e){ if (e.key === "Enter") $("addExpense").click(); });
$("filterCat").onchange = function(){ renderAll(); track("Filter Expenses", "select", this.value); };
$("clearBtn").onclick = function(){
  if (!data.expenses.length) return;
  if (!confirm("Delete all your expenses?")) return;
  data.expenses = []; save(); renderAll(); toast("All expenses cleared"); track("Clear Expenses", "button", "Clear all");
};

$("saveBudget").onclick = function(){
  var vals = {};
  for (var k = 0; k < BTYPES.length; k++){
    var raw = $("b_" + BTYPES[k]).value.trim();
    var v = raw === "" ? 0 : parseFloat(raw);
    if (!(v >= 0)){ toast("Enter a valid " + BTYPES[k] + " budget"); return; }
    vals[BTYPES[k]] = v;
  }
  data.budgets = vals;
  save(); renderAll(); toast("Budgets saved"); track("Set Budget", "form", "all");
};

function download(name, text, mime){
  var url = URL.createObjectURL(new Blob([text], {type: mime}));
  var a = document.createElement("a"); a.href = url; a.download = name;
  document.body.appendChild(a); a.click(); a.remove();
  setTimeout(function(){ URL.revokeObjectURL(url); }, 500);
}
$("csvBtn").onclick = function(){
  var q = function(s){ return '"' + String(s).replace(/"/g, '""') + '"'; };
  var csv = "date,title,category,amount\n" + data.expenses.map(function(e){ return [e.d, e.t, e.c, e.a].map(q).join(","); }).join("\n");
  download("expenses-" + user.id + ".csv", csv, "text/csv"); toast("CSV downloaded"); track("Download CSV", "button", "expenses");
};

/* ---------- analytics ---------- */
function renderCounts(counts){
  var box = $("actionCounts"), keys = Object.keys(counts);
  if (!keys.length){ box.innerHTML = '<div class="empty">No actions recorded yet.</div>'; return; }
  var max = Math.max.apply(null, keys.map(function(k){ return counts[k]; }));
  box.innerHTML = "";
  keys.sort(function(a, b){ return counts[b] - counts[a]; }).forEach(function(k){
    var r = el("div", "bar-row"), top = el("div", "bar-top");
    top.appendChild(el("span", "", k)); top.appendChild(el("span", "tag", counts[k] + (counts[k] === 1 ? " click" : " clicks")));
    var b = el("div", "bar"), i = el("i"); i.style.width = (counts[k] / max * 100) + "%";
    b.appendChild(i); r.appendChild(top); r.appendChild(b); box.appendChild(r);
  });
}
function renderActivity(box, events, n){
  box.innerHTML = "";
  if (!events.length){ box.appendChild(el("div", "empty", "No activity yet.")); return; }
  events.slice(-n).reverse().forEach(function(e){
    var it = el("div", "activity-item");
    it.appendChild(el("span", "", e.action + (e.user && e.user !== "-" ? " · " + e.user : "")));
    it.appendChild(el("small", "", e.screen + " · " + e.time));
    box.appendChild(it);
  });
}
function renderHeatmap(events){
  var box = $("heatmap"); box.innerHTML = "";
  if (!events.length){ box.appendChild(el("div", "empty", "No click activity yet.")); return; }
  events.slice(-150).forEach(function(e){
    if (e.x === "" || e.y === "") return;
    var d = el("div", "dot"); d.style.left = e.x + "%"; d.style.top = e.y + "%"; d.title = e.action + " (" + e.time + ")"; box.appendChild(d);
  });
}
function loadAnalytics(){
  fetch("/api/data").then(function(r){ return r.json(); }).then(function(d){
    $("totalClicks").textContent = d.total; $("uniqueActions").textContent = d.unique;
    $("userCount").textContent = d.users; $("sessionTime").textContent = d.seconds + "s";
    renderCounts(d.counts); renderActivity($("recentActivity"), d.events, 8);
    renderActivity($("activityFull"), d.events, 30); renderHeatmap(d.events);
  }).catch(function(){});
}
$("exportBtn").onclick = function(){
  fetch("/api/export").then(function(r){ return r.text(); }).then(function(t){
    download("usability-data.json", t, "application/json"); toast("JSON exported");
  });
};
$("resetBtn").onclick = function(){
  if (!confirm("Reset all tracking data?")) return;
  fetch("/api/reset", {method: "POST"}).then(function(){ toast("Tracking data reset"); loadAnalytics(); });
};

/* ---------- init ---------- */
(function(){
  var sel = $("eCat"), flt = $("filterCat");
  CATS.forEach(function(c){ var o = el("option", "", c); o.value = c; sel.appendChild(o); });
  var all = el("option", "", "All categories"); all.value = "All"; flt.appendChild(all);
  CATS.forEach(function(c){ var o = el("option", "", c); o.value = c; flt.appendChild(o); });
  var s = lsGet("sw:session", null);
  if (s && s.id){ user = s; showApp(); }
  setInterval(loadAnalytics, 1500);
})();
</script>
</body>
</html>
'''

def clean(s):
    return (str(s).replace("&", "&amp;").replace("<", "&lt;")
            .replace(">", "&gt;").replace('"', "&quot;").replace("\\", ""))

html_code = html_code.replace("__USER_NAME__", clean(USER_NAME))
HTML_FILE.write_text(html_code, encoding="utf-8")
print("Website created.")


# ================================================================
# COMPILE JAVA
# ================================================================
print("\nCompiling Java...")
res = subprocess.run([javac, "-encoding", "UTF-8", str(JAVA_FILE)],
                     cwd=str(PROJECT), capture_output=True, text=True)
if res.returncode != 0:
    print("\n❌ JAVA COMPILATION ERROR")
    print(res.stdout)
    print(res.stderr)
    raise RuntimeError("Java compilation failed.")
print("✅ Java compilation successful.")


# ================================================================
# FREE PORT + START SERVER
# ================================================================
def get_free_port():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(("127.0.0.1", 0))
    p = s.getsockname()[1]
    s.close()
    return p

PORT = get_free_port()
print("Free port:", PORT)

try:
    if "TRACKER_PROCESS" in globals() and TRACKER_PROCESS.poll() is None:
        TRACKER_PROCESS.terminate()
        time.sleep(1)
except Exception:
    pass

print("\nStarting Java server...")
log = open(LOG_FILE, "w")
TRACKER_PROCESS = subprocess.Popen(
    [java, "-Dtracker.port=" + str(PORT), "-Dtracker.root=" + str(PROJECT),
     "-cp", str(PROJECT), "SpendWiseServer"],
    cwd=str(PROJECT), stdout=log, stderr=subprocess.STDOUT, text=True)

server_ready = False
for _ in range(40):
    if TRACKER_PROCESS.poll() is not None:
        break
    try:
        r = urllib.request.urlopen("http://127.0.0.1:%d/" % PORT, timeout=1)
        if r.status == 200:
            server_ready = True
            break
    except Exception:
        time.sleep(0.5)

if not server_ready:
    print("\n❌ SERVER START FAILED")
    print(LOG_FILE.read_text()[-3000:])
    raise RuntimeError("Java server could not start.")
print("✅ Java server started successfully.")

url = output.eval_js("google.colab.kernel.proxyPort(%d)" % PORT)

print("\n" + "=" * 60)
print("🎉 YOUR APP IS READY")
print("=" * 60)

display(HTML(f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;max-width:560px;margin-top:16px;
            background:#14213d;color:#fff;border-radius:14px;padding:28px;
            box-shadow:0 10px 30px rgba(20,33,61,.25);">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:14px;">
    <div style="width:38px;height:38px;border-radius:10px;background:#2dd4bf;color:#14213d;
                display:flex;align-items:center;justify-content:center;font-weight:700;font-size:20px;">S</div>
    <div>
      <div style="font-size:20px;font-weight:700;">SpendWise &middot; Expense Tracker</div>
      <div style="font-size:13px;color:#b8c4dc;">Java backend running on port {PORT}</div>
    </div>
  </div>
  <div style="font-size:14px;color:#d5dded;margin-bottom:20px;">
    Developed by <b style="color:#fff;">{clean(USER_NAME)}</b>
  </div>
  <a href="{url}" target="_blank"
     style="display:inline-block;background:#2dd4bf;color:#14213d;padding:13px 26px;
            border-radius:9px;font-weight:700;font-size:15px;text-decoration:none;">
    Open SpendWise &#8599;
  </a>
</div>
"""))

SPENDWISE - EXPENSE TRACKER
Java : /usr/bin/java
Javac: /usr/bin/javac
Website created.

Compiling Java...
✅ Java compilation successful.
Free port: 42313

Starting Java server...
✅ Java server started successfully.

🎉 YOUR APP IS READY
